# 공개용 연구 노트북
선택 실행용 연구 기록입니다. 실행 전에 `../docs/reproducibility.md`를 읽고 모델·풀링·경로를 맞추세요. 원본 출력과 인증정보는 제거했습니다.


In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)


# RAG Evaluation Framework
This notebook implements a RAG framework using LangChain, Chroma, and Ollama, and evaluates it against the `allganize/RAG-Evaluation-Dataset-KO` dataset.

In [ ]:
# Install necessary packages if not already installed
%pip install langchain langchain-community langchain-chroma langchain-huggingface datasets pandas pypdf chromadb pymupdf pypdfium2 langchain-openai ragas pdfplumber sentence-transformers


In [ ]:
import json
import os
import pandas as pd

json_dir = r"data"

json_path = os.path.join(json_dir, "BioASQ-trainingDataset4b.json")

# 1) JSON 로드
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 2) 질문 리스트 꺼내기
# (BioASQ training은 보통 questions 키)
questions = data.get("questions") or data.get("question")

print("질문 수:", len(questions))

# 3) DataFrame 변환
df = pd.DataFrame(questions)

# 4) triples / concepts 제거
df = df.drop(columns=["triples", "concepts"], errors="ignore")

print(df.columns)
df


In [ ]:
df['type'].value_counts()

In [ ]:
# summary 제거
df = df[df["type"] != "summary"].reset_index(drop=True)

print("summary 제거 후 질문 수:", len(df))
print(df["type"].value_counts())


In [ ]:
prefix = "http://www.ncbi.nlm.nih.gov/pubmed/"

df["documents"] = df["documents"].apply(
    lambda lst: [x.replace(prefix, "") for x in lst] if isinstance(lst, list) else lst
)


In [ ]:
prefix = "http://www.ncbi.nlm.nih.gov/pubmed/"

def strip_snippet_docs(snips):
    if not isinstance(snips, list):
        return snips
    for s in snips:
        if isinstance(s, dict) and "document" in s and isinstance(s["document"], str):
            s["document"] = s["document"].replace(prefix, "")
    return snips

df["snippets"] = df["snippets"].apply(strip_snippet_docs)


In [ ]:
from pathlib import Path
yesno_df = df[df["type"] == "factoid"]
out_dir = Path(r"data")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "bioasq_factoid.csv"
yesno_df.to_csv(out_path, index=False, encoding="utf-8-sig")

In [ ]:
data = pd.read_csv("data/bioasq_list.csv")

In [ ]:
# 1) 데이터 전체에 NaN이 하나라도 있는지 (True/False)
print(data.isna().any().any())

# 2) 컬럼별 NaN 개수
print(data.isna().sum().sort_values(ascending=False))

# 3) NaN이 있는 행(row) 개수
print(data.isna().any(axis=1).sum())

# 4) NaN이 있는 행만 보고 싶을 때
print(data[data.isna().any(axis=1)])


In [ ]:
df['exact_answer'][0]

In [ ]:
len(df)

In [ ]:
df['type'].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import ast

# documents가 문자열로 저장돼 있을 수도 있어서 안전하게 list로 변환
def to_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if x == "" or x.lower() == "nan":
            return []
        try:
            v = ast.literal_eval(x)         # "[1,2,3]" -> [1,2,3]
            return v if isinstance(v, list) else [v]
        except Exception:
            # 혹시 literal_eval이 실패하면 콤마 split로 fallback
            return [t.strip() for t in x.split(",") if t.strip()]
    return [x]

docs_list = df["documents"].apply(to_list)
doc_cnt = docs_list.apply(len)   # 각 row에 문서 몇 개인지

# 1) 사분위수(Q1/Q2/Q3) 값 요약
q1, q2, q3 = doc_cnt.quantile([0.25, 0.5, 0.75]).tolist()
summary = {
    "count": int(doc_cnt.count()),
    "mean": float(doc_cnt.mean()),
    "min": int(doc_cnt.min()),
    "Q1(25%)": float(q1),
    "Q2(50%, median)": float(q2),
    "Q3(75%)": float(q3),
    "max": int(doc_cnt.max()),
    "IQR(Q3-Q1)": float(q3 - q1),
}
print(pd.Series(summary))



In [ ]:
df['documents'][0]

In [ ]:
df['snippets'][0]

In [ ]:
import re

def extract_pmid_from_url(url: str):
    if url is None:
        return None
    m = re.search(r"/pubmed/(\d+)", str(url))
    return m.group(1) if m else None

# documents 컬럼(각 행: URL list)을 전부 평탄화(flatten)해서 PMID 뽑기
all_pmids = set()

for doc_list in df["documents"]:
    if not isinstance(doc_list, list):
        continue
    for url in doc_list:
        pmid = extract_pmid_from_url(url)
        if pmid:
            all_pmids.add(pmid)

pmid_list = sorted(list(all_pmids))
print("고유 PMID 수:", len(pmid_list))
print("샘플:", pmid_list[:10])


In [ ]:
%pip install biopython

In [ ]:
# pubmed 크롤링
from Bio import Entrez
import pandas as pd
import time

Entrez.email = os.environ["NCBI_EMAIL"]
Entrez.tool = "MyPubMedScript"

def fetch_pubmed_title_abstract(pmid_list):
    titles, abstracts, pmids_done = [], [], []

    BATCH_SIZE = 50
    for i in range(0, len(pmid_list), BATCH_SIZE):
        batch_pmids = pmid_list[i:i+BATCH_SIZE]

        handle = Entrez.efetch(
            db="pubmed",
            id=",".join(batch_pmids),
            retmode="xml"
        )
        records = Entrez.read(handle)
        handle.close()

        for rec in records.get("PubmedArticle", []):
            pmid = str(rec["MedlineCitation"]["PMID"])
            pmids_done.append(pmid)

            # Title
            title = rec["MedlineCitation"]["Article"].get("ArticleTitle", None)

            # Abstract
            abstract = None
            try:
                abs_obj = rec["MedlineCitation"]["Article"]["Abstract"]["AbstractText"]
                # abs_obj는 list일 수도, str 비슷한 객체일 수도 있음
                if isinstance(abs_obj, list):
                    abstract = " ".join([str(x) for x in abs_obj])
                else:
                    abstract = str(abs_obj)
            except Exception:
                abstract = None

            titles.append(str(title) if title is not None else None)
            abstracts.append(abstract)

        time.sleep(0.34)  # NCBI rate limit 대비(너무 빠르면 막힘)

    return pd.DataFrame({"pmid": pmids_done, "title": titles, "abstract": abstracts})


In [ ]:
df_pubmed = fetch_pubmed_title_abstract(pmid_list)

# 결측치 제거(abstract 없으면 RAG에 쓸 가치 낮음)
df_pubmed = df_pubmed.dropna(subset=["abstract"]).reset_index(drop=True)

# 문서 텍스트 만들기
df_pubmed["text"] = df_pubmed.apply(
    lambda r: f"Title: {r['title']}\nAbstract: {r['abstract']}",
    axis=1
)

print(df_pubmed.head(3))

# 저장
df_pubmed.to_csv("pubmed_title_abstract.csv", index=False, encoding="utf-8")
print("저장 완료: pubmed_title_abstract.csv")


In [ ]:
df_pubmed

In [ ]:
df_pubmed['abstract'][0]

In [ ]:
import os
import pandas as pd
from datasets import load_dataset
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import load_prompt # 👈 core 모듈에서 가져오세요
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from tqdm import tqdm
import glob
from huggingface_hub import login

from sentence_transformers import SentenceTransformer

In [ ]:
import os
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)


## 3. Split Text (Chunking)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain.embeddings.base import Embeddings

In [ ]:
df = pd.read_csv(r"pubmed_title_abstract.csv")
df

In [ ]:
import pandas as pd

df = pd.read_csv(r"pubmed_title_abstract.csv")

text = df["text"].fillna("").astype(str)   # ✅ text 컬럼 직접 사용

char_len = text.str.len()
word_len = text.str.split().str.len()

def quartile_summary(s: pd.Series) -> pd.Series:
    q1, q2, q3 = s.quantile([0.25, 0.5, 0.75]).tolist()
    return pd.Series({
        "count": int(s.count()),
        "mean": float(s.mean()),
        "min": float(s.min()),
        "Q1(25%)": float(q1),
        "Q2(50%, median)": float(q2),
        "Q3(75%)": float(q3),
        "max": float(s.max()),
        "IQR(Q3-Q1)": float(q3 - q1),
    })

summary = pd.DataFrame({
    "chars": quartile_summary(char_len),
    "words": quartile_summary(word_len),
}).T

print(summary)


In [ ]:
import pandas as pd
df = pd.read_csv('pubmed_title_abstract.csv')
documents = df.to_dict(orient="records")

In [ ]:
documents

In [ ]:
document_list = [
    Document(
        page_content=d["text"],
        metadata={
            "pmid": str(d.get("pmid", "")),
            "title":  d.get("title", ""),
            "id":     d.get("id", i),        # 혹시 id 없으면 인덱스라도
        }
    )
    for i, d in enumerate(documents)
]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
splits = text_splitter.split_documents(document_list)
print(f"Created {len(splits)} chunks.")

In [ ]:
splits[10]

## 4. Create Vector Database (Chroma)

In [ ]:
from chromadb import PersistentClient

PERSIST_DIR = "chroma_db"

# Chroma 클라이언트 연결
client = PersistentClient(path=PERSIST_DIR)

# 전체 컬렉션 목록 조회
collections = client.list_collections()

print("📚 존재하는 Collection 목록:")
for c in collections:
    print(" -", c.name)

In [ ]:
from chromadb import PersistentClient

PERSIST_DIR = "500-100/embeddinggemma/chroma_db"

# Chroma 클라이언트 연결
client = PersistentClient(path=PERSIST_DIR)

# 전체 컬렉션 목록 조회
collections = client.list_collections()

print("📚 존재하는 Collection 목록:")
for c in collections:
    print(" -", c.name)

In [ ]:
%pip install -U accelerate

In [ ]:
# ============================================================
# BioASQ RAG Evaluator (single script, with main)
# total rag
# - uses existing Chroma (NO re-index / NO re-embedding)
# - embedding_function 생성 방식을 통일
# - Prompt: ✅ [SYSTEM]/[User] 분리 + SystemMessage/HumanMessage로 invoke (논문 Table 3)
# - ✅ 결과 CSV에 gold/pred exact & ideal을 함께 저장(비교용)
# ============================================================

import os, time, ast, json, re, unicodedata, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import ndcg_score

import torch
from torch import nn
from sentence_transformers import SentenceTransformer, models

from langchain_chroma import Chroma
from langchain_community.chat_models import ChatOllama

# (LangChain Embeddings import) - 버전 차이 대응
try:
    from langchain.embeddings.base import Embeddings
except Exception:
    from langchain_core.embeddings import Embeddings

# ✅ messages role 분리용
try:
    from langchain_core.messages import SystemMessage, HumanMessage
except Exception:
    from langchain.schema import SystemMessage, HumanMessage

# ✅ YAML 읽기
try:
    import yaml
except Exception as e:
    raise ImportError("pyyaml가 필요합니다. `pip install pyyaml` 후 다시 실행하세요.") from e


# =========================
# 0) 설정 (너 환경에 맞게)
# =========================
DATASET_LIST_CSV = "data/bioasq_factoid.csv"
CHROMA_DIR = "500-100/embeddinggemma/chroma_db"
PROMPT_PATH = "prompt/rag_prompt2.yaml"

RESULTS_DIR = "500-100/result"
SUMMARY_DIR = "500-100/result_summary"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

collection_list = ["gemma_C", "gemma_X","gemma_M" ]
k_list = [5,10,20]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LLM_MODEL = "gemma3:4b"
MAX_DOC_CHARS = 1000
MAX_TOTAL_CHARS = 40000

# LLM이 sources를 안 주면, 검색 결과를 evidence로 대체할지
EVIDENCE_FALLBACK_TO_RETRIEVAL = True

# ✅ 디버그를 콘솔 도배하지 않고 CSV로 저장
DEBUG_SAVE_FIRST_N = 50   # 처음 N개 질문만 디버그 저장(0이면 끔)

# =====================================================
# (선택) Seed 고정
# =====================================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# =========================
# 1) 통합 풀링 모듈
# =========================
class UnifiedConcatPooling(nn.Module):
    def __init__(self, word_embedding_dimension: int,
                 use_first=False, use_last=False, use_mean=False, use_max=False):
        super().__init__()
        self.word_embedding_dimension = word_embedding_dimension
        self.use_first = use_first
        self.use_last  = use_last
        self.use_mean  = use_mean
        self.use_max   = use_max

    def forward(self, features):
        token_embeddings = features["token_embeddings"]  # [B,T,H]
        attention_mask  = features["attention_mask"]     # [B,T]

        outs = []

        if self.use_first:
            outs.append(token_embeddings[:, 0])

        if self.use_last:
            last_idx = attention_mask.long().sum(dim=1) - 1
            last_idx = torch.clamp(last_idx, min=0)
            bidx = torch.arange(token_embeddings.size(0), device=token_embeddings.device)
            outs.append(token_embeddings[bidx, last_idx])

        if self.use_mean:
            mask = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)
            summed = (token_embeddings * mask).sum(dim=1)
            denom  = mask.sum(dim=1).clamp(min=1e-6)
            outs.append(summed / denom)

        if self.use_max:
            minus_inf = torch.finfo(token_embeddings.dtype).min
            masked = token_embeddings.masked_fill(attention_mask.eq(0).unsqueeze(-1), minus_inf)
            outs.append(masked.max(dim=1).values)

        if not outs:
            raise ValueError("At least one pooling must be enabled.")

        sentence_embedding = torch.cat(outs, dim=1) if len(outs) > 1 else outs[0]
        features["sentence_embedding"] = sentence_embedding
        return features

    def get_sentence_embedding_dimension(self):
        mult = int(self.use_first) + int(self.use_last) + int(self.use_mean) + int(self.use_max)
        return self.word_embedding_dimension * mult


# =========================
# 2) pooling 조건 맵
# =========================
CFG_MAP = {
    "C":  dict(first=True,  last=False, mean=False, max=False),
    "X":  dict(first=False, last=False, mean=False, max=True),
    "M":  dict(first=False, last=False, mean=True,  max=False),
    "CM": dict(first=True,  last=False, mean=True,  max=False),
    "CX": dict(first=True,  last=False, mean=False, max=True),
    "XM": dict(first=False, last=False, mean=True,  max=True),
}


# =========================
# 3) 컬렉션 prefix -> 모델 레지스트리
# =========================
MODEL_REGISTRY = {
    "SBERT": dict(model_name="sentence-transformers/all-MiniLM-L6-v2", is_decoder=False, force_append_eos=False),
    "bge":   dict(model_name="BAAI/bge-m3",                           is_decoder=False, force_append_eos=False),
    "e5":  dict(model_name="intfloat/multilingual-e5-large",                       is_decoder=False,  force_append_eos=False),
    "llama": dict(model_name="meta-llama/Llama-3.2-1B",               is_decoder=True,  force_append_eos=True),
   "bling": dict(model_name="Lajavaness/bilingual-embedding-large",               is_decoder=False,  force_append_eos = False),
  "solon": dict(model_name="OrdalieTech/Solon-embeddings-large-0.1",               is_decoder=False,  force_append_eos=False),
  "mxbai": dict(model_name="mixedbread-ai/mxbai-embed-large-v1",               is_decoder=False,  force_append_eos=False),
  "gemma": dict(model_name="google/embeddinggemma-300m",               is_decoder=False,  force_append_eos=False)
   
}


# =========================
# 4) SentenceTransformer/Wrapper 캐시
# =========================
_model_cache = {}
_wrapper_cache = {}


def parse_collection(collection_name: str):
    if "_" not in collection_name:
        raise ValueError(f"collection_name must contain '_' like 'bge_C': {collection_name}")
    prefix, cond = collection_name.rsplit("_", 1)
    if cond not in CFG_MAP:
        raise ValueError(f"Unknown pooling condition suffix={cond} in {collection_name}")
    if prefix not in MODEL_REGISTRY:
        raise ValueError(f"Unknown collection prefix={prefix} in {collection_name} (add to MODEL_REGISTRY)")
    return prefix, cond


def build_sentence_transformer(prefix: str, cond: str, device: str):
    key = (prefix, cond, device)
    if key in _model_cache:
        return _model_cache[key]

    info = MODEL_REGISTRY[prefix]
    model_name = info["model_name"]
    is_decoder = info["is_decoder"]

    model_args = {
        "trust_remote_code": True,
        "torch_dtype": torch.float16 if device == "cuda" else torch.float32,
    }
   
    word_embedding_model = models.Transformer(
        model_name_or_path=model_name,
        model_args=model_args,
        
    )

    tokenizer = word_embedding_model.tokenizer
    auto_model = word_embedding_model.auto_model

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            auto_model.resize_token_embeddings(len(tokenizer))

    auto_model.config.pad_token_id = tokenizer.pad_token_id

    cfg = CFG_MAP[cond].copy()

    if is_decoder and cfg["first"]:
        cfg["first"] = False
        cfg["last"] = True

    pooling_model = UnifiedConcatPooling(
        word_embedding_dimension=word_embedding_model.get_word_embedding_dimension(),
        use_first=cfg["first"],
        use_last=cfg["last"],
        use_mean=cfg["mean"],
        use_max=cfg["max"],
    )

    st = SentenceTransformer(modules=[word_embedding_model, pooling_model]).to(device)
    _model_cache[key] = (st, tokenizer, info["force_append_eos"])
    return _model_cache[key]


def make_wrapper_for_collection(collection_name: str, device: str) -> Embeddings:
    if collection_name in _wrapper_cache:
        return _wrapper_cache[collection_name]

    prefix, cond = parse_collection(collection_name)
    st, tokenizer, force_append_eos = build_sentence_transformer(prefix, cond, device)

    EOS = tokenizer.eos_token or ""

    def maybe_append_eos(text: str) -> str:
        if not force_append_eos or not EOS:
            return text
        text = "" if text is None else str(text)
        return text if text.endswith(EOS) else (text + EOS)

    class MyCustomModelWrapper(Embeddings):
        def __init__(self, model: SentenceTransformer, batch_size: int = 32, normalize: bool = False):
            self.model = model
            self.batch_size = batch_size
            self.normalize = normalize
            self.model.eval()

        def embed_documents(self, texts):
            texts = [maybe_append_eos(t) for t in texts]
            vecs = self.model.encode(
                texts, batch_size=self.batch_size,
                convert_to_numpy=True,
                normalize_embeddings=self.normalize,
                show_progress_bar=False,
            )
            return vecs.tolist()

        def embed_query(self, text):
            text = maybe_append_eos(text)
            vec = self.model.encode(
                [text],
                convert_to_numpy=True,
                normalize_embeddings=self.normalize,
                show_progress_bar=False,
            )[0]
            return vec.tolist()

    wrapper = MyCustomModelWrapper(st, batch_size=32, normalize=False)
    _wrapper_cache[collection_name] = wrapper
    return wrapper


# =========================
# 5) YAML prompt loader + ✅ [SYSTEM]/[User] 분리
# =========================
def load_prompt_text_from_yaml(path: str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Prompt file not found: {path}")

    raw = open(path, "r", encoding="utf-8").read()

    try:
        data = yaml.safe_load(raw)
    except Exception:
        return raw

    if isinstance(data, dict):
        for key in ["template", "prompt", "text", "content"]:
            if key in data and isinstance(data[key], str) and data[key].strip():
                return data[key]

        if "messages" in data and isinstance(data["messages"], list):
            parts = []
            for m in data["messages"]:
                if isinstance(m, dict):
                    c = m.get("content") or m.get("template") or ""
                    if isinstance(c, str) and c.strip():
                        parts.append(c.strip())
            if parts:
                return "\n\n".join(parts)

    if isinstance(data, list):
        parts = []
        for item in data:
            if isinstance(item, str) and item.strip():
                parts.append(item.strip())
            elif isinstance(item, dict):
                c = item.get("content") or item.get("template") or ""
                if isinstance(c, str) and c.strip():
                    parts.append(c.strip())
        if parts:
            return "\n\n".join(parts)

    return raw


def split_system_user(prompt_text: str):
    """
    prompt_text 안에 [SYSTEM] ... [User]/[USER] ... 가 있으면 분리.
    없으면 system="" / user=prompt_text 로 둔다.
    """
    t = prompt_text or ""
    if "[SYSTEM]" in t and "[User]" in t:
        sys_part = t.split("[SYSTEM]", 1)[1].split("[User]", 1)[0].strip()
        user_part = t.split("[User]", 1)[1].strip()
        return sys_part, user_part
    if "[SYSTEM]" in t and "[USER]" in t:
        sys_part = t.split("[SYSTEM]", 1)[1].split("[USER]", 1)[0].strip()
        user_part = t.split("[USER]", 1)[1].strip()
        return sys_part, user_part
    return "", t.strip()


# =========================
# 6) Evaluator
# =========================
class RAGEvaluator:
    def __init__(
        self,
        dataset_csv: str,
        collection_name: str,
        k: int,
        embedding_function: Embeddings,
        chroma_dir: str,
        llm_model: str = LLM_MODEL,
        prompt_path: str = PROMPT_PATH,
        do_llm: bool = True,
        save_contexts: bool = True,
    ):
        self.dataset_csv = dataset_csv
        self.collection_name = collection_name
        self.k = k
        self.embedding_function = embedding_function
        self.chroma_dir = chroma_dir
        self.llm_model = llm_model
        self.prompt_path = prompt_path
        self.do_llm = do_llm
        self.save_contexts = save_contexts

        dataset_name = os.path.splitext(os.path.basename(dataset_csv))[0]
        self.run_name = f"{dataset_name}__{self.collection_name}_k{self.k}"

        self.result_csv_path = os.path.join(RESULTS_DIR, self.run_name + ".csv")
        self.eval_csv_path = os.path.join(SUMMARY_DIR, self.run_name + "_retrieval.csv")
        self.debug_csv_path = os.path.join(SUMMARY_DIR, self.run_name + "_debug_retrieval.csv")

        self.df_questions = None
        self.df_results = None
        self.df_eval = None

        self._init_vectorstore()
        if self.do_llm:
            self._init_llm_and_prompt()

    # ---------- utils ----------
    @staticmethod
    def _maybe_parse(x):
        if isinstance(x, str):
            s = x.strip()
            if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
                try:
                    return ast.literal_eval(s)
                except Exception:
                    return x
        return x

    @staticmethod
    def _normalize(text):
        if text is None:
            return ""
        return unicodedata.normalize("NFC", str(text)).strip()

    @staticmethod
    def _unwrap_singleton(x, max_depth=3):
        # [[...]] 같은 “중첩 singleton list” 풀기
        for _ in range(max_depth):
            if isinstance(x, list) and len(x) == 1:
                x = x[0]
            else:
                break
        return x

    @staticmethod
    def _extract_pmid(value):
        if value is None:
            return None
        if isinstance(value, int):
            return str(value)
        if isinstance(value, dict):
            if "pmid" in value:
                return RAGEvaluator._extract_pmid(value["pmid"])
            if "document" in value:
                return RAGEvaluator._extract_pmid(value["document"])

        s = str(value).strip()
        if s.isdigit():
            return s

        patterns = [
            r"ncbi\.nlm\.nih\.gov/pubmed/(\d+)",
            r"www\.ncbi\.nlm\.nih\.gov/pubmed/(\d+)",
            r"pubmed\.ncbi\.nlm\.nih\.gov/(\d+)",
            r"/pubmed/(\d+)",
        ]
        for pat in patterns:
            m = re.search(pat, s)
            if m:
                return m.group(1)

        m = re.search(r"(\d+)$", s)
        if m and ("pubmed" in s or "ncbi" in s):
            return m.group(1)

        return None

    def _norm_pmid(self, pmid, source=None):
        p = self._extract_pmid(pmid)
        if not p and source:
            p = self._extract_pmid(source)
        return p

    @staticmethod
    def _safe_json(text: str):
        if text is None:
            return None
        t = text.strip()
        try:
            return json.loads(t)
        except Exception:
            pass
        m = re.search(r"\{.*\}", t, flags=re.DOTALL)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None

    @staticmethod
    def _find_offsets(full_text: str, span_text: str):
        if not full_text or not span_text:
            return None, None
        idx = full_text.find(span_text)
        if idx == -1:
            return None, None
        return idx, idx + len(span_text)

    # ✅ pred exact를 "gold와 같은 형태"로 맞춰 저장
    @staticmethod
    def _to_gold_format(qtype: str, pred_exact):
        qtype = (qtype or "").lower().strip()

        if qtype == "factoid":
            p = "" if pred_exact is None else str(pred_exact).strip()
            return [[p]] if p else []

        if qtype == "list":
            if not pred_exact:
                return []
            return [[str(x).strip()] for x in pred_exact if str(x).strip()]

        if qtype == "yesno":
            p = "" if pred_exact is None else str(pred_exact).strip().lower()
            return p if p in ["yes", "no"] else ""

        return "" if pred_exact is None else str(pred_exact).strip()

    # ---------- gold/pred helpers (BioASQ exact_answer 형태 대응) ----------
    @staticmethod
    def _gold_groups(exact_answer_field):
        x = RAGEvaluator._maybe_parse(exact_answer_field)
        groups = []

        if isinstance(x, list):
            for item in x:
                if isinstance(item, list):
                    s = {
                        RAGEvaluator._normalize(v).lower()
                        for v in item
                        if v is not None and RAGEvaluator._normalize(v)
                    }
                    if s:
                        groups.append(s)
                else:
                    v = RAGEvaluator._normalize(item).lower()
                    if v:
                        groups.append({v})
        elif isinstance(x, str):
            v = RAGEvaluator._normalize(x).lower()
            if v:
                groups.append({v})

        return groups

    @staticmethod
    def _pred_items(pred_answer):
        if pred_answer is None:
            return []
        if isinstance(pred_answer, list):
            return [RAGEvaluator._normalize(v).lower() for v in pred_answer if RAGEvaluator._normalize(v)]
        if isinstance(pred_answer, str):
            s = RAGEvaluator._normalize(pred_answer)
            if not s:
                return []
            parts = re.split(r"[;\n,]\s*", s)
            return [RAGEvaluator._normalize(p).lower() for p in parts if RAGEvaluator._normalize(p)]
        return []

    # list: group EM/F1
    def _list_group_f1(self, pred_answer, exact_answer_field):
        gold_groups = self._gold_groups(exact_answer_field)
        pred_items = self._pred_items(pred_answer)

        if not gold_groups and not pred_items:
            return 1.0
        if not gold_groups or not pred_items:
            return 0.0

        matched = 0
        used_pred = set()

        for g in gold_groups:
            hit = False
            for i, p in enumerate(pred_items):
                if i in used_pred:
                    continue
                if p in g:
                    hit = True
                    used_pred.add(i)
                    break
            if hit:
                matched += 1

        tp = matched
        precision = tp / len(pred_items)
        recall = tp / len(gold_groups)
        return 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)

    def _list_group_em(self, pred_answer, exact_answer_field):
        gold_groups = self._gold_groups(exact_answer_field)
        pred_items = self._pred_items(pred_answer)

        if not gold_groups and not pred_items:
            return 1
        if not gold_groups or not pred_items:
            return 0

        matched = 0
        used_pred = set()

        for g in gold_groups:
            hit = False
            for i, p in enumerate(pred_items):
                if i in used_pred:
                    continue
                if p in g:
                    hit = True
                    used_pred.add(i)
                    break
            if hit:
                matched += 1

        if matched == len(gold_groups) and len(used_pred) == len(pred_items):
            return 1
        return 0

    # ---------- init vectorstore ----------
    def _init_vectorstore(self):
        self.vectorstore = Chroma(
            collection_name=self.collection_name,
            persist_directory=self.chroma_dir,
            embedding_function=self.embedding_function,
        )
        self.retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": self.k},
        )
        print(f"✅ Chroma 로드 완료: {self.collection_name} (k={self.k})")

    # ---------- init llm + prompt ----------
    def _init_llm_and_prompt(self):
        print(f"--- LLM 로드: ChatOllama({self.llm_model}) ---")
        self.llm = ChatOllama(model=self.llm_model)
        print("✅ LLM 로드 완료")

        print(f"--- Prompt YAML 로드: {self.prompt_path} ---")
        self.prompt_text = load_prompt_text_from_yaml(self.prompt_path)
        if not isinstance(self.prompt_text, str) or not self.prompt_text.strip():
            raise ValueError("Prompt YAML에서 유효한 프롬프트 텍스트를 찾지 못했습니다.")

        # ✅ [SYSTEM]/[User] 분리 (없으면 user로 취급)
        self.system_tmpl, self.user_tmpl = split_system_user(self.prompt_text)

        # user 파트가 아예 없으면, 최소 user 템플릿을 깔아줌
        if not self.user_tmpl.strip():
            self.user_tmpl = "[Question Type] question: {question}\nAnswer:"

        print("✅ Prompt 로드 완료 (system/user split)")

    # ---------- load ----------
    def load_questions(self):
        self.df_questions = pd.read_csv(self.dataset_csv)

        # ✅ snippets는 안 쓰므로 파싱/필수 컬럼에서 제거
        for col in ["documents", "exact_answer"]:
            if col in self.df_questions.columns:
                self.df_questions[col] = self.df_questions[col].apply(self._maybe_parse)

        need_cols = ["body", "documents", "type", "id", "exact_answer"]
        missing = [c for c in need_cols if c not in self.df_questions.columns]
        if missing:
            raise ValueError(f"질문 CSV에 필요한 컬럼이 없음: {missing}")

        if "ideal_answer" not in self.df_questions.columns:
            self.df_questions["ideal_answer"] = ""

        print(f"✅ 질문 로드 완료: {len(self.df_questions)} rows")

    # ---------- context builder ----------
    def _build_context(self, retrieved_docs):
        blocks = []
        docid_to_text = {}
        docid_to_pmid = {}
        total = 0

        for rank, doc in enumerate(retrieved_docs):
            pmid = self._norm_pmid(doc.metadata.get("pmid"), doc.metadata.get("source")) or "UNKNOWN"

            chunk_text = (doc.page_content or "").strip()
            if len(chunk_text) > MAX_DOC_CHARS:
                chunk_text = chunk_text[:MAX_DOC_CHARS]

            doc_id = f"{pmid}#{rank}"
            block = f"[DOCID={doc_id}][PMID={pmid}]\n{chunk_text}\n"

            if total + len(block) > MAX_TOTAL_CHARS:
                break

            blocks.append(block)
            total += len(block)

            docid_to_text[doc_id] = chunk_text
            docid_to_pmid[doc_id] = pmid

        return "\n".join(blocks), docid_to_text, docid_to_pmid

    # ✅ system/user 각각에 치환
    def _format_tmpl(self, tmpl: str, context_str: str, question: str, qtype: str) -> str:
        p = tmpl or ""
        if "{context}" in p:
            p = p.replace("{context}", context_str)
        if "{question}" in p:
            p = p.replace("{question}", question)
        if "{type}" in p:
            p = p.replace("{type}", qtype)
        return p

    # ---------- parse LLM output ----------
    def _parse_llm_output(self, raw_text: str):
        parsed = self._safe_json(raw_text)
        pred_exact = ""
        pred_ideal = ""
        sources = []

        if isinstance(parsed, dict):
            pred_exact = parsed.get("exact_answer", "")
            if pred_exact == "" and "answer" in parsed:
                pred_exact = parsed.get("answer", "")

            pred_ideal = parsed.get("ideal_answer", "")

            src = parsed.get("sources", []) or []
            if isinstance(src, list):
                sources = src

            return pred_exact, pred_ideal, sources

        t = (raw_text or "").strip()
        return (t if t else ""), "", []

    # ✅ 타입별 exact sanitize (factoid/list에서 yes/no 방지)
    def _sanitize_pred_exact_by_type(self, qtype: str, pred_exact):
        qtype = (qtype or "").lower().strip()

        # 중첩 singleton 풀기
        pred_exact = self._unwrap_singleton(pred_exact)

        # 공통: 문자열이면 strip
        if isinstance(pred_exact, str):
            s = pred_exact.strip()
        else:
            s = None

        # ✅ factoid인데 yes/no면 무조건 빈값
        if qtype == "factoid":
            if isinstance(pred_exact, list):
                pred_exact = pred_exact[0] if pred_exact else ""
                pred_exact = "" if pred_exact is None else str(pred_exact).strip()
            else:
                pred_exact = "" if pred_exact is None else str(pred_exact).strip()

            if pred_exact.lower() in ["yes", "no"]:
                pred_exact = ""  # 핵심 가드
            return pred_exact

        # ✅ list인데 yes/no면 []
        if qtype == "list":
            if pred_exact is None:
                return []
            if isinstance(pred_exact, str):
                if pred_exact.strip().lower() in ["yes", "no"]:
                    return []
            # 아래에서 list 형태로 강제
            if isinstance(pred_exact, str):
                s = pred_exact.strip()
                if s == "":
                    return []
                if s.startswith("[") and s.endswith("]"):
                    try:
                        tmp = json.loads(s)
                        if isinstance(tmp, list):
                            return tmp
                    except Exception:
                        pass
                return [a for a in re.split(r"[;\n,]\s*", s) if a]
            if isinstance(pred_exact, list):
                # 리스트 내부도 정리
                out = []
                for v in pred_exact:
                    vv = "" if v is None else str(v).strip()
                    if vv:
                        out.append(vv)
                return out
            return [str(pred_exact).strip()] if str(pred_exact).strip() else []

        # yesno
        if qtype == "yesno":
            if isinstance(pred_exact, list):
                pred_exact = pred_exact[0] if pred_exact else ""
            pred_exact = "" if pred_exact is None else str(pred_exact).strip().lower()
            return pred_exact if pred_exact in ["yes", "no"] else ""

        # 기타
        if isinstance(pred_exact, list):
            pred_exact = pred_exact[0] if pred_exact else ""
        return "" if pred_exact is None else str(pred_exact).strip()

    # ---------- generation ----------
    def run_generation(self):
        if self.df_questions is None:
            self.load_questions()

        results = []
        debug_rows = []
        start = time.time()

        for idx, (_, row) in enumerate(tqdm(self.df_questions.iterrows(), total=len(self.df_questions), desc="Retrieve+Generate")):
            question = self._normalize(row.get("body", ""))
            qtype = self._normalize(row.get("type", "")).lower()

            gold_exact = row.get("exact_answer")
            gold_ideal = row.get("ideal_answer", "")

            # 1) retrieve
            retrieved_docs = self.retriever.invoke(question)

            retrieved_pmids = []
            retrieved_docids = []
            contexts = []

            for rank, doc in enumerate(retrieved_docs):
                pmid = self._norm_pmid(doc.metadata.get("pmid"), doc.metadata.get("source")) or "UNKNOWN"
                retrieved_pmids.append(pmid)
                retrieved_docids.append(f"{pmid}#{rank}")
                if self.save_contexts:
                    contexts.append(doc.page_content)

            # 2) generation
            pred_exact_answer_goldfmt = []
            pred_ideal_answer = ""
            sources_out = []
            raw = ""

            # 내부 평가용
            generated_answer = None
            ideal_answer = ""

            if self.do_llm:
                context_str, docid_to_text, docid_to_pmid = self._build_context(retrieved_docs)

                # ✅ system/user 각각 치환 후 messages로 invoke
                sys_msg = self._format_tmpl(self.system_tmpl, context_str, question, qtype)
                usr_msg = self._format_tmpl(self.user_tmpl, context_str, question, qtype)

                try:
                    msg = self.llm.invoke([
                        SystemMessage(content=sys_msg),
                        HumanMessage(content=usr_msg),
                    ])
                    raw = msg.content if hasattr(msg, "content") else str(msg)
                except Exception as e:
                    raw = ""
                    print(f"\n[WARN] LLM invoke error: {repr(e)}")

                pred_exact, pred_ideal, sources_in = self._parse_llm_output(raw)

                # ✅ 타입별 sanitize (factoid에서 yes/no 차단 포함)
                pred_exact = self._sanitize_pred_exact_by_type(qtype, pred_exact)
                pred_ideal = "" if pred_ideal is None else str(pred_ideal).strip()

                generated_answer = pred_exact
                ideal_answer = pred_ideal

                # ✅ 저장용: gold 형태로
                pred_exact_answer_goldfmt = self._to_gold_format(qtype, pred_exact)
                pred_ideal_answer = pred_ideal

                # ---------- sources 검증(있을 때만) ----------
                if isinstance(sources_in, list):
                    for s in sources_in:
                        if not isinstance(s, dict):
                            continue

                        doc_id = s.get("doc_id")
                        pmid = s.get("pmid")
                        quote = s.get("quote")

                        if quote is None and isinstance(s.get("text"), str):
                            quote = s.get("text")

                        doc_id = str(doc_id) if doc_id is not None else None
                        pmid = str(pmid) if pmid is not None else None
                        quote = quote if isinstance(quote, str) else None

                        if not doc_id or doc_id not in docid_to_text:
                            continue

                        chunk_text = docid_to_text.get(doc_id, "")
                        pmid_expected = docid_to_pmid.get(doc_id)

                        pmid_norm = self._extract_pmid(pmid) if pmid else None
                        pmid_expected_norm = self._extract_pmid(pmid_expected) if pmid_expected else None

                        if not pmid_norm:
                            pmid_norm = pmid_expected_norm
                        if pmid_expected_norm and pmid_norm != pmid_expected_norm:
                            continue

                        if quote is None:
                            continue

                        b, e = self._find_offsets(chunk_text, quote)
                        if b is None or e is None:
                            continue

                        sources_out.append({
                            "doc_id": doc_id,
                            "pmid": pmid_norm,
                            "text": quote,
                            "offsetInBeginSection": b,
                            "offsetInEndSection": e,
                        })

                if EVIDENCE_FALLBACK_TO_RETRIEVAL and not sources_out:
                    sources_out = [
                        {
                            "doc_id": f"{p}#{i}",
                            "pmid": p,
                            "text": None,
                            "offsetInBeginSection": None,
                            "offsetInEndSection": None,
                        }
                        for i, p in enumerate(retrieved_pmids)
                        if p and p != "UNKNOWN"
                    ]

            # 3) answer metrics (임시/참고용)
            if qtype == "list":
                answer_em = float(self._list_group_em(generated_answer, gold_exact))
                answer_f1 = float(self._list_group_f1(generated_answer, gold_exact))

            elif qtype == "factoid":
                gold_groups = self._gold_groups(gold_exact)
                pred = self._normalize(generated_answer).lower()
                hit = 0.0
                if pred and gold_groups:
                    for g in gold_groups:
                        if pred in g:
                            hit = 1.0
                            break
                answer_em = hit
                answer_f1 = hit

            elif qtype == "yesno":
                gold = self._normalize(gold_exact).lower()
                pred = self._normalize(generated_answer).lower()
                hit = 1.0 if (gold in ["yes", "no"] and pred == gold) else 0.0
                answer_em = hit
                answer_f1 = hit

            else:
                answer_em = np.nan
                answer_f1 = np.nan

            out = {
                "body": row.get("body"),
                "documents": row.get("documents"),
                "type": row.get("type"),
                "id": row.get("id"),

                # gold
                "gold_exact_answer": gold_exact,
                "gold_ideal_answer": gold_ideal,

                # retrieval
                "retrieved_pmids": retrieved_pmids,
                "retrieved_docids": retrieved_docids,

                # pred
                "pred_exact_answer": pred_exact_answer_goldfmt,  # ✅ 하나만 저장(골드형태)
                "pred_ideal_answer": pred_ideal_answer,

                "raw_llm": raw,
                "sources": sources_out,

                "answer_em": float(answer_em) if answer_em is not np.nan else np.nan,
                "answer_f1": float(answer_f1) if answer_f1 is not np.nan else np.nan,
            }
            if self.save_contexts:
                out["contexts"] = contexts

            results.append(out)

            if DEBUG_SAVE_FIRST_N and idx < DEBUG_SAVE_FIRST_N:
                gold_docs = row.get("documents", []) or []
                gold_pmids = [self._extract_pmid(d) for d in gold_docs]
                gold_pmids = [p for p in gold_pmids if p]

                overlap = set(gold_pmids) & set([p for p in retrieved_pmids if p and p != "UNKNOWN"])
                unk_ratio = sum(p == "UNKNOWN" for p in retrieved_pmids) / max(len(retrieved_pmids), 1)

                debug_rows.append({
                    "qid": row.get("id"),
                    "qtype": qtype,
                    "question_head": question[:120],
                    "gold_count": len(set(gold_pmids)),
                    "retrieved_count": len(set(retrieved_pmids)),
                    "overlap_count": len(overlap),
                    "overlap_example": ";".join(list(overlap)[:5]),
                    "gold_pmids_head": ";".join(gold_pmids[:10]),
                    "retrieved_pmids_head": ";".join([p for p in retrieved_pmids[:10]]),
                    "unknown_ratio": float(unk_ratio),
                })

        self.df_results = pd.DataFrame(results)
        self.df_results.to_csv(self.result_csv_path, index=False, encoding="utf-8-sig")
        print(f"📁 결과 저장: {self.result_csv_path} (time={time.time()-start:.1f}s)")

        if debug_rows:
            pd.DataFrame(debug_rows).to_csv(self.debug_csv_path, index=False, encoding="utf-8-sig")
            print(f"📁 debug 저장: {self.debug_csv_path}")

    # ---------- evidence doc-level metrics ----------
    def _evidence_doc_scores(self, gold_documents, pred_sources):
        gold = set()
        for d in (gold_documents or []):
            pmid = self._extract_pmid(d)
            if pmid:
                gold.add(pmid)

        pred = set()
        for s in (pred_sources or []):
            if isinstance(s, dict) and s.get("pmid"):
                p = self._extract_pmid(s.get("pmid"))
                if p:
                    pred.add(p)

        if not gold:
            return np.nan, np.nan, np.nan

        inter = gold & pred
        prec = len(inter) / max(len(pred), 1)
        rec = len(inter) / len(gold)
        hit = 1.0 if len(inter) > 0 else 0.0
        return prec, rec, hit

    # ---------- retrieval evaluation ----------
    def run_retrieval_eval(self):
        if self.df_results is None:
            self.df_results = pd.read_csv(self.result_csv_path)

        df = self.df_results.copy()

        for c in ["documents", "retrieved_pmids", "retrieved_docids",
                  "gold_exact_answer", "gold_ideal_answer",
                  "pred_exact_answer", "pred_ideal_answer",
                  "sources", "contexts"]:
            if c in df.columns:
                df[c] = df[c].apply(self._maybe_parse)

        eval_rows = []
        for _, row in df.iterrows():
            gold_set = set()
            docs = row.get("documents", None)
            if isinstance(docs, list):
                for d in docs:
                    pmid = self._extract_pmid(d)
                    if pmid:
                        gold_set.add(pmid)

            pred = row.get("retrieved_pmids", [])
            pred_norm = []
            if isinstance(pred, list):
                seen = set()
                for p in pred:
                    pm = self._extract_pmid(p) or (str(p).strip() if p is not None else None)
                    if not pm or pm == "UNKNOWN":
                        continue
                    if pm not in seen:
                        pred_norm.append(pm)
                        seen.add(pm)

            L = min(self.k, len(pred_norm))
            pred_top = pred_norm[:L]

            if len(gold_set) == 0 or L == 0:
                recall_k = np.nan
                precision_k = np.nan
                hit_k = np.nan
                ndcg_k = np.nan
            else:
                hit_count = sum([1 for p in pred_top if p in gold_set])
                recall_k = hit_count / len(gold_set)
                precision_k = hit_count / L
                hit_k = 1.0 if hit_count > 0 else 0.0

                rel = [1 if (p in gold_set) else 0 for p in pred_top]
                if sum(rel) == 0:
                    ndcg_k = 0.0
                else:
                    L2 = len(rel)
                    K2 = max(self.k, 2)
                    rel_pad = (rel + [0.0] * (K2 - L2))[:K2]
                    score_pad = (list(range(L2, 0, -1)) + [0.0] * (K2 - L2))[:K2]
                    y_true  = np.asarray([rel_pad], dtype=float)
                    y_score = np.asarray([score_pad], dtype=float)
                    ndcg_k = float(ndcg_score(y_true, y_score, k=min(self.k, K2)))

            ev_prec, ev_rec, ev_hit = self._evidence_doc_scores(
                row.get("documents", []),
                row.get("sources", []),
            )

            eval_rows.append({
                "id": row.get("id"),
                "type": row.get("type"),
                "recall@k": recall_k,
                "precision@k": precision_k,
                "ndcg@k": ndcg_k,
                "hit@k": hit_k,

                "answer_em": row.get("answer_em", np.nan),
                "answer_f1": row.get("answer_f1", np.nan),

                "evidence_precision": ev_prec,
                "evidence_recall": ev_rec,
                "evidence_hit": ev_hit,

                "gold_doc_count": len(gold_set),
            })

        self.df_eval = pd.DataFrame(eval_rows)
        self.df_eval.to_csv(self.eval_csv_path, index=False, encoding="utf-8-sig")
        print(f"📁 retrieval 평가 저장: {self.eval_csv_path}")

        overall_recall = float(self.df_eval["recall@k"].mean(skipna=True))
        overall_precision = float(self.df_eval["precision@k"].mean(skipna=True))
        overall_ndcg = float(self.df_eval["ndcg@k"].mean(skipna=True))
        overall_hit = float(self.df_eval["hit@k"].mean(skipna=True))

        overall_answer_em = float(self.df_eval["answer_em"].mean(skipna=True))
        overall_answer_f1 = float(self.df_eval["answer_f1"].mean(skipna=True))

        overall_ev_precision = float(self.df_eval["evidence_precision"].mean(skipna=True))
        overall_ev_recall = float(self.df_eval["evidence_recall"].mean(skipna=True))
        overall_ev_hit = float(self.df_eval["evidence_hit"].mean(skipna=True))

        print(f"- Recall@K          : {overall_recall:.6f}")
        print(f"- Precision@K       : {overall_precision:.6f}")
        print(f"- NDCG@K            : {overall_ndcg:.6f}")
        print(f"- Hit@K             : {overall_hit:.6f}")
        print(f"- AnswerEM          : {overall_answer_em:.6f}")
        print(f"- AnswerF1          : {overall_answer_f1:.6f}")
        print(f"- EvidencePrecision : {overall_ev_precision:.6f}")
        print(f"- EvidenceRecall    : {overall_ev_recall:.6f}")
        print(f"- EvidenceHit       : {overall_ev_hit:.6f}")

    def run_all(self):
        self.run_generation()
        self.run_retrieval_eval()


# =========================
# 7) main
# =========================
if __name__ == "__main__":
    for col in collection_list:
        wrapper = make_wrapper_for_collection(col, device=DEVICE)

        for k in k_list:
            print(f"\n=== Running | dataset={os.path.basename(DATASET_LIST_CSV)} | collection={col} | k={k} ===")
            evaluator = RAGEvaluator(
                dataset_csv=DATASET_LIST_CSV,
                collection_name=col,
                k=k,
                embedding_function=wrapper,
                chroma_dir=CHROMA_DIR,
                do_llm=True,
                save_contexts=True,
            )
            evaluator.run_all()
            print("****************************************\n")


In [ ]:
import gc, torch

# (선택) 네 캐시 dict가 있을 때만
try:
    _wrapper_cache.clear()
    _model_cache.clear()
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # (선택) 멀티프로세스/IPC 캐시 정리


In [ ]:

# 2-3. 나머지 배치를 tqdm으로 감싸 순차적으로 추가
for i, batch in enumerate(tqdm(batches, desc="Creating Vector Store Batches", unit="batch")):
    # Chroma의 add_documents를 사용하여 문서 추가
    vectorstore.add_documents(
        documents=batch,
        collection_name=COLLECTION_NAME
    )

total_creation_time = time.time() - total_start_time
print(f"\n✅ 벡터 저장소 생성 완료! 총 소요 시간: {total_creation_time:.2f}초")
print(f"저장소에 최종 저장된 문서 수: {vectorstore._collection.count()}")

In [ ]:
# Create Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
!pip install colab-xterm
%load_ext colabxterm

In [ ]:
%xterm


In [ ]:
!ollama list

## 5. Setup RAG Pipeline

In [ ]:
# Initialize LLM
llm = ChatOllama(model="gemma3:4b")

# Pull Prompt
prompt = load_prompt("prompt/rag_prompt.yaml")

# Define Post-processing to extract source info
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 6. Evaluation Loop

In [ ]:
import pandas as pd
import os
from tqdm import tqdm
# 필요한 라이브러리 임포트 (retriever, llm, prompt 등은 이미 정의되어 있다고 가정)

# 테스트용 (전체 데이터 실행 시 주석 해제)
# df = df[:10]

results = []

for index, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
    question = row['question']

    # 1. 문서 검색 (Retrieve)
    retrieved_docs = retriever.invoke(question)

    # 2. [핵심 추가] 검색된 본문 내용 추출 (List[str] 형태)
    # RAGAS의 'faithfulness', 'context_precision' 평가는 이 'contexts' 컬럼을 필요로 합니다.
    retrieved_contexts = [doc.page_content for doc in retrieved_docs]

    # 3. LLM 생성을 위한 컨텍스트 포맷팅
    context_str = format_docs(retrieved_docs)

    # 4. 답변 생성 (Generate Answer)
    rag_chain_with_context = (
        prompt
        | llm
        | StrOutputParser()
    )

    generated_answer = rag_chain_with_context.invoke({"context": context_str, "question": question})

    #print(generated_answer)
    #print(retrieved_docs)
    # 5. 메타데이터 추출 (파일명, 페이지)
    retrieved_files = [os.path.basename(doc.metadata.get('source', 'Unknown')) for doc in retrieved_docs]
    retrieved_pages = [doc.metadata.get('page', -1) for doc in retrieved_docs]

    # 6. 결과 저장
    result_row = row.to_dict()
    result_row['generated_answer'] = generated_answer
    result_row['contexts'] = retrieved_contexts  # <--- 검색된 본문 내용 리스트 추가
    result_row['retrieved_file_name'] = retrieved_files
    result_row['retrieved_page_no'] = retrieved_pages

    results.append(result_row)

# Create Result DataFrame
result_df = pd.DataFrame(results)
result_df.head()
result_df.to_csv("results/bge_rag_evaluation_results_fium_1000_k6.csv", index=False)

In [ ]:
#check result
import pandas as pd
df = pd.read_csv("500-100/result/bioasq_factoid__bge_CM_k20.csv")
df.head()

In [ ]:
df['contexts'][0]

In [ ]:
# #check total result
import os, glob, re, ast
import numpy as np
import pandas as pd

RESULT_DIR = r"500-100\result"

def parse_filename(fname: str):
    base = os.path.basename(fname).strip()
    m = re.match(
        r"^bioasq_(?P<qtype>factoid|list|yesno)[_]+(?P<model>[^_]+)_(?P<pool>[^_]+)_k(?P<k>\d+)\.csv$",
        base,
        flags=re.IGNORECASE
    )
    if not m:
        return {"qtype": None, "model": None, "pool": None, "k": None}
    d = m.groupdict()
    d["qtype"] = d["qtype"].lower()
    d["k"] = int(d["k"])
    return d

def read_csv_smart(path):
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, engine="python")

def to_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if s == "":
        return []
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            return to_list(ast.literal_eval(s))
        except Exception:
            pass
    return [s]

def norm_text(t):
    return re.sub(r"\s+", " ", str(t).strip().lower())

# ---------- Factoid MRR ----------
def compute_factoid_mrr(df: pd.DataFrame) -> float:
    for c in ["factoid_mrr", "mrr", "MRR", "exact_mrr"]:
        if c in df.columns:
            return float(pd.to_numeric(df[c], errors="coerce").mean())

    gold_candidates = ["gold_exact", "gold_exact_answer", "exact_answer", "gold"]
    pred_candidates = ["pred_exact", "pred_exact_answer", "pred"]

    gold_col = next((c for c in gold_candidates if c in df.columns), None)
    pred_col = next((c for c in pred_candidates if c in df.columns), None)
    if gold_col is None or pred_col is None:
        return np.nan

    mrrs = []
    for _, row in df.iterrows():
        gold_list = [norm_text(x) for x in to_list(row[gold_col])]
        pred_list = [norm_text(x) for x in to_list(row[pred_col])]
        if not pred_list:
            mrrs.append(0.0); continue

        ranks = []
        for g in gold_list:
            for i, p in enumerate(pred_list, start=1):
                if g == p:
                    ranks.append(i); break
        mrrs.append(1.0 / min(ranks) if ranks else 0.0)

    return float(np.mean(mrrs)) if mrrs else np.nan

# ---------- List F-measure ----------
def compute_list_f(df: pd.DataFrame) -> float:
    for c in ["list_f_measure", "f_measure", "F_measure", "f1", "F1", "list_f1"]:
        if c in df.columns:
            return float(pd.to_numeric(df[c], errors="coerce").mean())

    gold_candidates = ["gold_exact", "gold_exact_answer", "exact_answer", "gold"]
    pred_candidates = ["pred_exact", "pred_exact_answer", "pred"]

    gold_col = next((c for c in gold_candidates if c in df.columns), None)
    pred_col = next((c for c in pred_candidates if c in df.columns), None)
    if gold_col is None or pred_col is None:
        return np.nan

    f1s = []
    for _, row in df.iterrows():
        gold = set(norm_text(x) for x in to_list(row[gold_col]) if str(x).strip() != "")
        pred = set(norm_text(x) for x in to_list(row[pred_col]) if str(x).strip() != "")

        if not gold and not pred:
            f1s.append(1.0); continue
        if not gold or not pred:
            f1s.append(0.0); continue

        tp = len(gold & pred)
        prec = tp / len(pred)
        rec  = tp / len(gold)
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(f1)

    return float(np.mean(f1s)) if f1s else np.nan

# ---------- Yes/No Macro F1 ----------
def compute_yesno_macro_f1(df: pd.DataFrame) -> float:
    # 이미 집계된 컬럼이 있으면 우선 사용
    for c in ["yesno_macro_f1", "yesno_f1", "macro_f1", "MacroF1", "exact_yesno_macro_f1"]:
        if c in df.columns:
            return float(pd.to_numeric(df[c], errors="coerce").mean())

    gold_candidates = ["gold_exact", "gold_exact_answer", "exact_answer", "gold"]
    pred_candidates = ["pred_exact", "pred_exact_answer", "pred"]

    gold_col = next((c for c in gold_candidates if c in df.columns), None)
    pred_col = next((c for c in pred_candidates if c in df.columns), None)
    if gold_col is None or pred_col is None:
        return np.nan

    # gold/pred를 라벨(yes/no)로 정규화
    def as_label(v):
        lst = to_list(v)
        if not lst:
            return ""
        s = norm_text(lst[0])
        if "yes" in s:
            return "yes"
        if "no" in s:
            return "no"
        return s  # 혹시 다른 표기가 있으면 그대로

    y_true = [as_label(v) for v in df[gold_col].tolist()]
    y_pred = [as_label(v) for v in df[pred_col].tolist()]

    labels = ["yes", "no"]
    f1s = []
    for lab in labels:
        tp = sum((t == lab and p == lab) for t, p in zip(y_true, y_pred))
        fp = sum((t != lab and p == lab) for t, p in zip(y_true, y_pred))
        fn = sum((t == lab and p != lab) for t, p in zip(y_true, y_pred))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(f1)

    return float(np.mean(f1s)) if f1s else np.nan

# ---------- summary 생성 ----------
csv_files = sorted(glob.glob(os.path.join(RESULT_DIR, "*.csv")))
rows = []

for fp in csv_files:
    df = read_csv_smart(fp)
    meta = parse_filename(os.path.basename(fp))
    qtype = meta["qtype"]

    rows.append({
        "file": os.path.basename(fp),
        "qtype": qtype,
        "model": meta["model"],
        "pool": meta["pool"],
        "k": meta["k"],
        "factoid_mrr": compute_factoid_mrr(df) if qtype == "factoid" else np.nan,
        "list_f_measure": compute_list_f(df) if qtype == "list" else np.nan,
        "yesno_macro_f1": compute_yesno_macro_f1(df) if qtype == "yesno" else np.nan,
        "n_rows": len(df),
    })

summary = pd.DataFrame(rows)

bad = summary[summary["qtype"].isna()][["file"]]
if len(bad) > 0:
    print("⚠️ 파일명 파싱 실패 목록:")
    print(bad.to_string(index=False))

# ---------- pivots ----------
fact = summary[summary["qtype"] == "factoid"].copy()
fact_pivot = fact.pivot_table(index=["model","pool"], columns="k", values="factoid_mrr", aggfunc="mean").sort_index()

lst = summary[summary["qtype"] == "list"].copy()
list_pivot = lst.pivot_table(index=["model","pool"], columns="k", values="list_f_measure", aggfunc="mean").sort_index()

yn = summary[summary["qtype"] == "yesno"].copy()
yesno_pivot = yn.pivot_table(index=["model","pool"], columns="k", values="yesno_macro_f1", aggfunc="mean").sort_index()

print("=== FACTOID (MRR) PIVOT ===")
display(fact_pivot)

print("\n=== LIST (F-measure) PIVOT ===")
display(list_pivot)

print("\n=== YESNO (Macro F1) PIVOT ===")
display(yesno_pivot)
